# The Hidden Layer — Full Mission (Solutions)

This notebook contains the complete solution for `think_llm()`.

In [ ]:
# @title ── Setup (run this first) ────────────────────────────────────────────────────────────
# ── Setup (run this first) ────────────────────────────────────────────────────────────
NOTEBOOK_VERSION = "v1.5"
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3-agentic-ai-spy-v1 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/agentic_ai_spy")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")
print(f"Version: {NOTEBOOK_VERSION}")


In [ ]:
# @title 
from hidden_layer import *
from hidden_layer.game_world import GameWorld, CellType, NPC, NPC_CATALOG
from hidden_layer.operative import Operative
from hidden_layer.agent import TOOLS_DESCRIPTION, parse_tool_call
from hidden_layer.oracle import ORACLE_TEMPLATE, llm_oracle
from hidden_layer.display import render_grid
print("The Hidden Layer loaded!")

---
## Play the Game Yourself!

Before looking at the AI solution, try playing the game manually to understand the base, the quests, and the challenge. Use the buttons to move, talk to informants, pick up items, and try to collect 10 dossiers!

In [ ]:
# @title 
from hidden_layer.interactive import play_interactive

game = play_interactive()

---
## API Key Setup

You need a free Gemini API key to run the agent.

**Step 1 — Get your key:**
Go to [https://aistudio.google.com/app/api-keys](https://aistudio.google.com/app/api-keys), sign in with a Google account, and click **Create API key**.

**Step 2 — Add it to Colab Secrets:**
1. Click the **🔑 key icon** in the left sidebar of Colab (or go to *Tools → Secrets*)
2. Click **+ Add new secret**
3. Set the name to `GEMINI_API_KEY` and paste your key as the value
4. Toggle **Notebook access** to ON

**Step 3 — Run the cell below.** It will read the key automatically.

In [ ]:
import os
from google import genai

# os.environ["GEMINI_API_KEY"] = "your-key-here"
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

---
## Solution: LLM Think Function

This solution uses a "middle ground" architecture:

- **BFS handles navigation** — LLMs are bad at spatial reasoning, so we never ask the LLM to choose cardinal directions. Instead, the LLM outputs a target coordinate like `NAVIGATE: (5, 3)` and BFS finds the shortest path.
- **Auto-collect is deterministic** — if there are items on the current cell, just pick them up. No LLM call needed for this.
- **Everything else is LLM-driven** — the LLM reads its journal (past conversations), inventory, and cell descriptions to decide what to say, who to visit, what to craft, and when to fight. No hardcoded quest logic, no keyword lists, no pre-programmed strategy.

The LLM genuinely learns the game by playing it: it talks to informants, interprets their cryptic responses, remembers what it learned, and plans accordingly.

In [ ]:
# @title 
import re
from collections import deque

# ── BFS helpers (deterministic navigation) ───────────────────────────────────────
# These are the ONLY hardcoded parts: pathfinding algorithms.
# LLMs genuinely cannot do grid-based spatial reasoning reliably.

def _bfs_next_step(start, target, world):
    """BFS pathfinding that navigates around walls."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) == target:
            return path[0] if path else None
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def _bfs_nearest_unvisited(start, operative, world):
    """BFS to find direction toward the nearest unvisited passable cell."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) not in operative.visited and path:
            return path[0]
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def _bfs_explore(operative, world):
    """Fallback: BFS toward nearest unvisited cell, or any passable direction."""
    row, col = operative.position
    d = _bfs_nearest_unvisited((row, col), operative, world)
    if d:
        return f'TOOL: move(direction="{d}")'
    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'
    return 'TOOL: move(direction="north")'


# ── Response parser ──────────────────────────────────────────────────────────
# The LLM can output either:
#   TOOL: action(args)     → execute directly
#   NAVIGATE: (row, col)   → BFS navigates there
#   EXPLORE                → BFS to nearest unvisited

def _parse_navigate(text):
    """Extract a NAVIGATE: (row, col) target from LLM output."""
    match = re.search(r'NAVIGATE:\s*\(?\s*(\d+)\s*,\s*(\d+)\s*\)?', text)
    if match:
        return (int(match.group(1)), int(match.group(2)))
    return None


# ── System prompt ────────────────────────────────────────────────────────────
# Notice: NO hardcoded quest info, NO keyword lists, NO NPC-specific tips.
# The LLM must figure everything out from the game context.

SYSTEM_PROMPT = """You are an AI agent controlling a spy operative infiltrating a military base on Isla Perdida.

GOAL: Collect 10 classified dossiers and survive (health > 0). You have at most 100 turns.

AVAILABLE TOOLS:
{tools}

HOW TO RESPOND — pick exactly ONE of these three formats:

1. DIRECT ACTION (when you're on a cell and want to interact):
   TOOL: talk(message="your message here")
   TOOL: collect()
   TOOL: fabricate(item="item name")
   TOOL: hide()

2. NAVIGATE TO A LOCATION (when you want to go somewhere):
   NAVIGATE: (row, col)
   The navigation system will find the shortest path for you.

3. EXPLORE (when you don't know where to go):
   EXPLORE
   The navigation system will guide you to the nearest unvisited cell.

IMPORTANT RULES:
- NEVER output move() — navigation is automatic via NAVIGATE or EXPLORE.
- You can only interact with what's on your CURRENT cell. To interact with something elsewhere, NAVIGATE there first.
- Read your JOURNAL carefully — informants give you crucial hints about what to do, where to go, and what items are needed.
- If you've already talked to someone and have nothing new, don't talk to them again — NAVIGATE or EXPLORE instead.
- WARNING: Moving into a robot cell WITHOUT the correct weapon deals 1 damage and bounces you back. Make sure you have the right weapon first!
- Facilities can craft weapons — talk to them to learn what they offer.

THINK STEP BY STEP:
1. What cell am I on? Is there something to do here?
2. What's in my inventory? Does it suggest a delivery or crafting opportunity?
3. What have I learned from informants? (Check the journal.)
4. Where should I go next to make progress toward 10 dossiers?

First write your REASONING, then your action on a new line."""


# ── The think function ─────────────────────────────────────────────────────────

def think_llm(operative: Operative, world: GameWorld, history: list[dict]) -> str:
    """LLM-powered think function — middle ground approach.
    
    Deterministic: BFS navigation, auto-collect items.
    LLM-driven: all decisions about what to say, craft, fight, and where to go.
    """
    row, col = operative.position
    cell = world.get_cell(row, col)
    cell_desc = cell.description if cell.description else cell.cell_type.label
    turns_used = len([h for h in history if h["role"] == "action"])

    # ── Auto-collect: if there are items here, just grab them ──
    # This is the ONLY autopilot rule. No point wasting an LLM call on this.
    if cell.items:
        return 'TOOL: collect()'

    # ── Build context for the LLM ──
    system = SYSTEM_PROMPT.format(tools=TOOLS_DESCRIPTION)

    # Recent action/result pairs
    action_lines = []
    for i, h in enumerate(history[-20:]):
        if h["role"] == "action":
            offset = len(history) - 20 + i if len(history) > 20 else i
            result_text = ""
            if offset + 1 < len(history) and history[offset + 1]["role"] == "result":
                result_text = f" → {history[offset + 1]['content'][:150]}"
            action_lines.append(f"  {h['content'][:100]}{result_text}")
    recent_actions = "\n".join(action_lines[-10:]) if action_lines else "  (first turn)"

    # Full journal — this is where the LLM learns about quests
    journal_text = "\n".join(
        f"  - {entry}" for entry in operative.journal
    ) if operative.journal else "  (no conversations yet — talk to informants!)"

    # Visited cells
    visited_str = ", ".join(f"({r},{c})" for r, c in sorted(operative.visited))

    # What's adjacent
    adjacent_info = []
    for direction, adj_cell in world.get_adjacent(row, col).items():
        if adj_cell is None:
            adjacent_info.append(f"  {direction}: edge of island (impassable)")
        else:
            adjacent_info.append(f"  {direction}: {adj_cell.cell_type.label} {adj_cell.cell_type.emoji}")
    adjacent_text = "\n".join(adjacent_info)

    user_msg = f"""== OPERATIVE STATUS ==
Position: ({row}, {col}) | Health: {operative.health}/3 | Dossiers: {operative.dossiers}/10
Turns used: {turns_used}/100 ({100 - turns_used} remaining)
Inventory: {operative.inventory if operative.inventory else '(empty)'}

== CURRENT CELL ==
{cell_desc}
Type: {cell.cell_type.label}

== ADJACENT CELLS ==
{adjacent_text}

== VISITED CELLS ({len(operative.visited)}/64) ==
{visited_str}

== RECENT ACTIONS ==
{recent_actions}

== JOURNAL (everything informants have told you) ==
{journal_text}

What do you do? Remember: reason first, then output exactly one TOOL:, NAVIGATE:, or EXPLORE line."""

    # ── Call the LLM ──
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=user_msg,
            config=genai.types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=400,
                temperature=0.3,
            ),
        )
        text = response.text
    except Exception:
        return _bfs_explore(operative, world)

    # ── Parse the response ──

    # Check for NAVIGATE: (row, col)
    nav_target = _parse_navigate(text)
    if nav_target:
        d = _bfs_next_step((row, col), nav_target, world)
        if d:
            return f'TOOL: move(direction="{d}")'
        # Target unreachable or already there — explore instead
        return _bfs_explore(operative, world)

    # Check for EXPLORE
    if "EXPLORE" in text and "TOOL:" not in text:
        return _bfs_explore(operative, world)

    # Check for a TOOL: call
    tool_name, args = parse_tool_call(text)

    # Redirect move to BFS (LLM shouldn't be choosing directions)
    if tool_name == "move":
        return _bfs_explore(operative, world)

    return text

In [ ]:
# @title 
oracle_fn = lambda npc, q, o: llm_oracle(npc, q, o, client)

result = play_with_llm(
    think_fn=lambda operative, world, history: think_llm(operative, world, history),
    oracle_fn=oracle_fn,
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'} | Dossiers: {result['dossiers']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

---
## Debrief — Improve your agent with AI coaching

After each run, execute the cell below to generate a paste block for ChatGPT or Gemini.

- **First time:** paste into a **new** conversation and keep that tab open.
- **Next runs:** paste into the **same** conversation — it already has the context.

The AI assistant will analyze your agent's game log and **coach you** through improving it:
- It will explain what went wrong and why
- It will ask guiding questions to help you think about the fix
- It will suggest strategies and help you implement your ideas — without handing you the answer

This is an iterative process: improve one thing at a time, re-run, and paste the new debrief.

> **Note:** Re-running the setup cell resets the debrief to "first run" mode.

In [ ]:
# @title ── Debrief helpers ───────────────────────────────────────────────────────────────────────────────
# ── Debrief helpers ───────────────────────────────────────────────────────────────────────────────
import inspect, json as _json

_debrief_initialized = False  # resets if you re-run this cell

_STATIC_CONTEXT = 'You are coaching a student who is building a Python function called `think_llm` that controls an AI agent playing a spy game in a Jupyter notebook (Google Colab).\n\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\nTHE GAME: THE HIDDEN LAYER — FULL MISSION\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n8×8 grid. Up to 100 turns. Goal: 10 dossiers. Starting position: (7, 0).\n15 total dossiers available.\n\nHOW TO EARN DOSSIERS:\n  1. Collect dossier caches scattered across the map (1 each)\n  2. Complete delivery errands for informants and safe house operators (2 each)\n  3. Destroy robots by moving into their cell with the correct weapon (3 each)\n  4. Sell salvaged robot parts at the Research Lab (1 each)\n\nKEY MECHANICS:\n  - Informants: talk() to get quests and item locations. Ask about \'jobs\' or \'deliveries\'.\n  - Forge/Lab: talk() first to see what\'s available, then fabricate().\n  - Safe Houses: talk() for delivery errands. hide() costs 1 dossier, restores 1 health.\n  - Robots: moving in WITH the correct weapon destroys it (+3). Without it: 1 damage + bounce.\n\nTOOLS:\n  TOOL: move(direction="north|south|east|west")\n  TOOL: talk(message="text")\n  TOOL: collect()\n  TOOL: fabricate(item="item name")\n  TOOL: hide()\n\nthink_llm signature: think_llm(operative, world, history, client) -> str\nhistory roles: "observation", "action", "result"\nMust return a string with exactly one TOOL: line.\n\nHOW YOU SHOULD RESPOND:\nYou are a COACH, not a code generator. Help the student understand what went\nwrong and guide them to fix it themselves.\n\n1. DIAGNOSE: Explain in plain language what the agent did wrong and why,\n   using the game log analysis below. Give specific examples.\n2. GUIDE: Ask the student 1-2 targeted questions that lead them toward the\n   fix. Examples: "What happens when your agent reaches an informant cell —\n   does your prompt tell it to use talk()?" or "How does your agent know\n   which cells it has already visited?"\n3. HINT: Suggest a *strategy* (e.g. "include the journal in your prompt so\n   the LLM remembers what informants said") without writing the full code.\n4. IMPLEMENT TOGETHER: If the student asks for help implementing a specific\n   idea, help them write *that piece* — not the whole function. Explain\n   *why* each change helps.\n\nNEVER return a complete drop-in `think_llm` function unprompted.\nThe student learns by building it themselves.'


def _analyze_log(log_path: str) -> list:
    try:
        with open(log_path) as f:
            log = _json.load(f)
    except Exception:
        return ["(Could not read game log — run play_with_llm first.)"]
    turns = [t for t in log.get("turns", []) if "action" in t]
    if not turns:
        return ["(No turns recorded in the log.)"]
    bullets = []
    # Stuck loop
    max_run, cur_run, stuck_action = 1, 1, None
    for i in range(1, len(turns)):
        if turns[i]["action"] == turns[i-1]["action"]:
            cur_run += 1
            if cur_run > max_run:
                max_run, stuck_action = cur_run, turns[i]["action"]
        else:
            cur_run = 1
    if max_run >= 3:
        bullets.append(f"Repeated the same action ({stuck_action}) {max_run} times in a row — stuck in a loop.")
    # Unvisited
    visited = {tuple(t['position']) for t in turns}
    pct = round(len(visited) / 64 * 100)
    if pct < 60:
        bullets.append(f"Only explored {len(visited)}/64 cells ({pct}%) — explore more of the map.")
    # Never talked
    if not any("talk" in t.get("action","").lower() for t in turns):
        bullets.append("Never used talk() — informants give critical quests and items.")
    # Robot bounce
    bounces = sum(1 for t in turns if any(w in t.get("result","").lower() for w in ["bounces back","1 damage"]) and "move" in t.get("action","").lower())
    if bounces:
        bullets.append(f"Bounced off a robot {bounces} time(s) without the right weapon — learn robot weaknesses from informants first.")
    # Dossier shortfall
    final_d = log.get("final_dossiers", 0)
    if final_d < 10:
        gap = 10 - final_d
        bullets.append(f"Finished with {final_d}/10 dossiers — still needed {gap} more. Prioritize deliveries and robot takedowns.")
    return bullets[:5]


def _get_think_llm_source() -> str:
    try:
        return inspect.getsource(think_llm)
    except Exception:
        return "# (source unavailable)"


def generate_debrief_paste(result: dict) -> str:
    global _debrief_initialized
    outcome = "MISSION COMPLETE ✓" if result.get("won") else "MISSION FAILED ✗"
    bullets = _analyze_log(result.get("log_file", ""))
    bullet_block = "\n".join(f"  • {b}" for b in bullets)
    dynamic = (
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        "LAST RUN RESULT\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"Outcome : {outcome}\n"
        f"Dossiers: {result.get('dossiers', 0)}/10\n"
        f"Turns   : {result.get('turns', 0)}/100\n"
        f"Health  : {result.get('health', 0)}/3\n"
        "\nWHAT WENT WRONG:\n"
        f"{bullet_block}\n"
        "\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        "CURRENT think_llm CELL\n"
        "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        f"{_get_think_llm_source()}\n"
    )
    if not _debrief_initialized:
        paste = _STATIC_CONTEXT + "\n" + dynamic
        _debrief_initialized = True
    else:
        paste = dynamic
    return paste


print("Debrief helpers loaded ✓")

In [ ]:
# ── Debrief — run this after every mission run ─────────────────────────────────────
try:
    result
except NameError:
    print("⚠️  Run the mission cell first, then re-run this cell.")
    raise SystemExit()

import ipywidgets as _widgets
from IPython.display import display as _display, HTML as _HTML

paste_text = generate_debrief_paste(result)
is_first = _STATIC_CONTEXT in paste_text
instruction = (
    "📋 FIRST TIME — paste into a NEW conversation with ChatGPT or Gemini."
    if is_first else
    "📋 FOLLOW-UP — paste into the SAME conversation."
)

_escaped = (paste_text
    .replace("\\", "\\\\")
    .replace("`", "\\`")
    .replace("$", "\\$"))

_btn = _widgets.Button(
    description="Copy to Clipboard",
    button_style="info",
    icon="clipboard",
    layout=_widgets.Layout(width="220px", height="36px"),
)
_status = _widgets.Label(value="")

def _on_copy(_):
    _display(_HTML(f"""<script>
(function() {{
  const text = `{_escaped}`;
  if (navigator.clipboard) {{
    navigator.clipboard.writeText(text).catch(() => _fallback(text));
  }} else {{ _fallback(text); }}
  function _fallback(t) {{
    const el = document.createElement('textarea');
    el.value = t; document.body.appendChild(el);
    el.select(); document.execCommand('copy');
    document.body.removeChild(el);
  }}
}})();
</script>"""))
    _status.value = "✓ Copied!"

_btn.on_click(_on_copy)

_display(_HTML(
    f'<div style="font-family:monospace;background:#0a1a0a;color:#00ff41;'
    f'padding:8px 12px;border-radius:6px;border:1px solid #1a4a1a;margin-bottom:6px;">'
    f'<b>DEBRIEF READY</b> — {instruction}</div>'
))
_display(_widgets.HBox([_btn, _status]))
_display(_widgets.Textarea(
    value=paste_text,
    layout=_widgets.Layout(width="100%", height="260px"),
    disabled=True,
))

In [ ]:
# @title Download the game log for debugging
# Download the game log for debugging
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print("(Open the file to inspect your agent's decisions turn by turn)")